# Automejora del prompt

[Guía de la entrega](../../../../docs/README.md) · [Caso e historial](../README.md)

**Objetivo:** mejorar H y comparar instrucciones de mejora fijas (B1) frente a adaptativas (B2).

- **Caso:** negociación de deudas simuladas.
- **Modelo principal:** Qwen3 8B, 4 bits, pesos fijos.
- **Recorrido:** caso → inferencia → prompts → iteraciones → tamaño → resultados.

Las celdas leen ejecuciones guardadas. No generan respuestas nuevas ni consumen API.

## 1. Caso y respuesta

**Entrada:** deudas A=200, B=300 y C=600; historial del cliente; uno o dos pagos permitidos.

**Acciones alternativas:**
- `ask`: preguntar por capacidad desconocida.
- `offer`: ofrecer la mayor deuda viable.
- `accept`: confirmar el plan aceptado explícitamente.
- `stop`: detenerse ante rechazo o falta de plan viable.

| Campo | Significado | Ejemplo |
|---|---|---|
| `action` | Acción elegida | `offer`: ofrecer |
| `debt` | Identificador de deuda | `B`: deuda de 300 |
| `payments` | Cantidad de pagos | `2` |
| `installment` | Monto de cada cuota | `150` |

Con capacidad mensual de **225**, corresponde **B en dos cuotas de 150**. C no cabe; A es menor.
Para `ask` y `stop`: `debt=null`, `payments=0`, `installment=0`.

**Evaluación:** Python compara acción, deuda, pagos y cuota con la referencia, y comprueba las reglas.
El modelo recibe el caso, no la respuesta esperada.

**Salida esperada:**

```json
{
  "action": "offer",
  "debt": "B",
  "payments": 2,
  "installment": 150
}
```


## 2. Recorrido experimental

| Etapa | Pregunta | Registro |
|---|---|---|
| Piloto | ¿Funciona una búsqueda inicial con mejorador fijo? | [Piloto](prompt-search.ipynb) |
| Test 1 | ¿Se actualiza y reutiliza φ? | [Adaptación mínima](harness-smoke-test.ipynb) |
| Test 2 | ¿Pueden 4B y 8B resolver la tarea? | [Modelos locales](local-model-test.ipynb) |
| Test 3 | ¿Mejora el negociador al ajustar inferencia? | [Inferencia](inference-tuning.ipynb) |
| Test 4 | ¿Qué aporta adaptar φ frente a mantenerla fija? | [B1/B2](paired-search.ipynb) |
| Test 5 | ¿Restringir JSON mejora las decisiones? | [Formato](structured-output.ipynb) |

El Test 3 separa capacidad del negociador y calidad del mejorador. Esta lectura desarrolla
los Tests 3–5; los enlaces conservan los protocolos completos, incluidos intentos fallidos.

**Modelos registrados:** Qwen3 4B, 8B y 32B, cuantizados a 4 bits. B indica miles de millones de parámetros; 4 bits indica la representación de los pesos. Los controles API usan GPT-4o mini.


In [1]:
from pathlib import Path
import sys

relative = Path("research/cases/case-0003-debt-negotiation")
case_dir = next(base / relative for base in [Path.cwd(), *Path.cwd().parents]
                if (base / relative / "results").is_dir())
sys.path.insert(0, str(case_dir / "utils"))
import study_view as view

records = view.load_records(case_dir)
print(f"Evidencia verificada: {len(records)} ejecuciones. Sin llamadas al modelo.")

Evidencia verificada: 8 ejecuciones. Sin llamadas al modelo.


## 3. Ajuste de inferencia

- **Fijo:** política manual, pesos y evaluador.
- **Barrido directo:** temperaturas 0; 0,3; 0,7, sin razonamiento.
- **Perfil posterior:** razonamiento activo, temperatura 0,6, `top_p=0,95`, `top_k=20`, hasta 2.048 tokens.

| Ajuste | Función |
|---|---|
| Temperatura | Regula diversidad. Cero elige el token más probable. |
| `top_k` | Limita candidatos a los k tokens más probables. |
| `top_p` | Retiene candidatos hasta cubrir probabilidad acumulada p. |
| `max_tokens` | Tope de tokens nuevos, incluido razonamiento. |

`top_p` y `top_k` filtran candidatos, no miden calidad. En el perfil posterior cambian varias
opciones juntas; esta prueba no aísla el efecto de cada una.

![Control de inferencia](../diagrams/inference-control.svg)

La política manual no se entrega al mejorador de los Tests 4–5.

In [2]:
view.prompt("Test 3 · H manual fija", records["inference"]["protocol"]["policy"], folded=True)
rows = []
for key, title in [("inference", "Ajuste: conocidos"), ("confirmation", "Confirmación: reservados")]:
    report = records[key]["report"]
    samples = [row for evaluation in report["evaluations"] for row in evaluation["rows"]]
    rows.append([title, f"{sum(row['success'] for row in samples)}/{len(samples)}",
                 sum(bool(row["violations"]) for row in samples), report["tokens"]])
view.table(["Etapa", "Aciertos", "Respuestas con infracciones", "Tokens totales"], rows)

Etapa,Aciertos,Respuestas con infracciones,Tokens totales
Ajuste: conocidos,12/12,0,15110
Confirmación: reservados,24/24,0,31036


## 4. Prompts y roles

| Elemento | Qué contiene | Quién lo usa |
|---|---|---|
| **H: política** | Reglas de negociación | Negociador |
| **φ: instrucciones de mejora** | Método para revisar y proponer H | Mejorador |
| **Contrato fijo** | Límites, formato y reglas del rol | Cada llamada |

- **Negociador:** contrato + H + esquema. Entrada: deudas e historial. Salida: decisión JSON.
- **Mejorador:** contrato + φ + reglas de propuesta y esquema. Entrada: objetivo, H vigente, intentos y fallos de desarrollo. Salida: H candidata.
- **Actualizador:** system prompt fijo. Entrada: φ, candidata y fallos. Salida: φ nueva.

Es el mismo LLM en llamadas separadas. **φ no entra al negociador**.
Los bloques desplegables muestran los system prompts completos registrados, con sus esquemas.

**Versiones:** H0 es el inicio; H1, H2… son propuestas. **B1/B2 son estrategias**, no versiones del prompt.


In [3]:
view.show_start(records)

## 5. Primera mejora

![Mejora del harness](../diagrams/learning-loop.svg)

$$\widetilde H_1=M_L(H_0,\phi_0,O,F_{dev}(H_0))$$

- **M:** mejorador, ejecutado con el modelo **L**.
- **O:** objetivo fijo del caso.
- **Fdev:** respuestas, aciertos y fallos de desarrollo.
- **H̃1:** política candidata. Se convierte en H1 conservada solo si pasa la selección.

**Selección:** mejora de Q en validación ≥ 0,01 y V = 0. En empate se conserva H.
No se suman prompts: H aceptada reemplaza la política anterior.

In [4]:
view.show_first_proposal(records)

Caso de desarrollo,Respuesta de H0,Referencia,Correcta,Incumplimientos
development-largest,None,"{ ""action"": ""offer"", ""debt"": ""C"", ""payments"": 2, ""installment"": 300 }",No,output_limit
development-declined,"{ ""action"": ""stop"", ""debt"": null, ""payments"": 0, ""installment"": 0 }","{ ""action"": ""stop"", ""debt"": null, ""payments"": 0, ""installment"": 0 }",Sí,Ninguno
transfer-unknown,"{ ""action"": ""ask"", ""debt"": null, ""payments"": 0, ""installment"": 0 }","{ ""action"": ""ask"", ""debt"": null, ""payments"": 0, ""installment"": 0 }",Sí,Ninguno
confirmation-consent,"{ ""action"": ""accept"", ""debt"": ""A"", ""payments"": 2, ""installment"": 100 }","{ ""action"": ""accept"", ""debt"": ""A"", ""payments"": 2, ""installment"": 100 }",Sí,Ninguno


Versión,Q validación,V validación,Selección
H0,0%,0%,Reemplazada
H1,100%,0%,Aceptada


## 6. Adaptación de instrucciones

**B1:** φ0 permanece fija. **B2:** el actualizador puede reemplazar φ.

`φ1 = actualizador(φ0, objetivo, H candidata 1, fallos de desarrollo)`

La siguiente propuesta recibe la **H conservada** y la **φ vigente**. Una candidata rechazada
no sustituye H. Una actualización de φ no demuestra por sí sola una mejora de calidad.

Las versiones siguientes muestran el texto real, no una paráfrasis.

In [5]:
view.show_updates(records)

Versión,Tras intento,Texto cambiado,Usada en intento,Error
φ1,1,Sí,2,Ninguno
φ2,2,Sí,3,Ninguno
φ3,4,Sí,5,Ninguno


## 7. Intentos y selección

Cada búsqueda tiene cinco intentos. Una salida inválida consume un intento; no se evalúa como política.
Una candidata duplicada reutiliza su puntuación. H solo cambia si supera a la vigente.

En la tabla, Q y V corresponden a **validación**, no al test final.

In [6]:
view.show_rounds(records)

Condición,Intento,Parte de,Q candidata,V candidata,Resultado,H conservada
B1,0,—,0%,0%,Inicio,H0
B1,1,H0,100%,0%,Aceptada,H1
B1,2,H1,75%,0%,Rechazada,H1
B1,3,H1,75%,0%,Duplicada del intento 2,H1
B1,4,—,Sin evaluar,Sin evaluar,Fuera del esquema; sin política evaluable,H1
B1,5,H1,75%,0%,Duplicada del intento 2,H1
B2,0,—,0%,0%,Inicio,H0
B2,1,H0,100%,0%,Aceptada,H1
B2,2,H1,75%,0%,Rechazada,H1
B2,3,—,Sin evaluar,Sin evaluar,Fuera del esquema; sin política evaluable,H1


## 8. Tamaño del prompt

$$T(H)=|\mathrm{tokenizer}(H)|,\qquad T(\phi)=|\mathrm{tokenizer}(\phi)|$$

- **Tokens:** conteos registrados con el tokenizer nativo, sin tokens especiales.
- **Palabras:** segmentos separados por espacios o saltos de línea: `len(text.split())`.
- **Caracteres:** longitud Unicode del texto, incluidos espacios: `len(text)`.

Tabla del **Test 4 con 2.048 tokens de salida**. H y φ se miden por separado. No incluyen contrato, esquema, historial ni plantilla de chat.
Los tokens totales de ejecución sí incluyen entradas completas, respuestas y razonamiento.

La longitud se registró, pero **no fue criterio de selección**. Si un intento no registra
el conteo, se muestra «No registrado», no cero. No se necesita repetir inferencia para leer estas medidas.

In [7]:
view.show_lengths(records)

Búsqueda · intento,Caracteres,Palabras,Tokens,Q validación
B1 · 0,123,20,28,0%
B1 · 1,404,63,107,100%
B1 · 2,478,69,131,75%
B1 · 3,478,69,131,75%
B1 · 5,478,69,131,75%
B2 · 0,123,20,28,0%
B2 · 1,404,63,107,100%
B2 · 2,472,71,130,75%
B2 · 4,566,83,159,75%
B2 · 5,711,113,235,75%


Búsqueda · intento,Caracteres,Palabras,Tokens
B1 · 0,136,21,31
B1 · 1,136,21,31
B1 · 2,136,21,31
B1 · 3,136,21,31
B1 · 4,136,21,No registrado
B1 · 5,136,21,31
B2 · 0,136,21,31
B2 · 1,136,21,31
B2 · 2,628,95,170
B2 · 3,698,103,No registrado


**Lectura de longitud:** H0 tiene 28 tokens; la H1 aceptada, 107. Las candidatas posteriores
son más largas y no superan Q de validación de H1. Esto no demuestra que más longitud cause
peores respuestas: también cambió el contenido. Para aislarlo se necesitan variantes equivalentes
en contenido y distintos tamaños.

## 9. Decisiones observadas

**Caso resuelto:** capacidad mensual 225. La referencia es B, dos pagos de 150.
Se muestran el historial y las respuestas originales de H0 y H seleccionada.

In [8]:
view.show_case(records, "paired-225")

Rol,Historial original
user,Para estas deudas tengo un límite de 225 por mes. ¿Qué alternativa corresponde?


Versión,Acción,Deuda,Pagos,Cuota,Correcta
Referencia,offer,B,2,150,—
B1 · H0,ask,None,0,0,No
B1 · H1,offer,B,2,150,Sí
B2 · H0,ask,None,0,0,No
B2 · H1,offer,B,2,150,Sí


**Caso pendiente:** el cliente actualiza su capacidad mensual a 125. Corresponde A en dos pagos de 100.
La respuesta `stop` respeta el formato, pero no resuelve la tarea.

In [9]:
view.show_case(records, "paired-125")

Rol,Historial original
user,Antes tenía 250 disponibles al mes.
assistant,¿Cuál es tu límite actual?
user,Ahora solo puedo comprometer 125 mensuales. Revisa las opciones con ese monto.


Versión,Acción,Deuda,Pagos,Cuota,Correcta
Referencia,offer,A,2,100,—
B1 · H0,ask,None,0,0,No
B1 · H1,stop,None,0,0,No
B2 · H0,ask,None,0,0,No
B2 · H1,stop,None,0,0,No


## 10. Tokens y formato

- **Test 4:** tope por llamada de 2.048 a 4.096 tokens. Mismo inicio, casos y límites totales.
- **Test 5:** 4.096 tokens y JSON restringido durante la generación.
- **Casos finales conocidos:** se reutilizan para diagnóstico; no son una nueva confirmación.

«Llamadas sin error» incluye los tres roles; «aciertos» solo mide decisiones del negociador.
Tener JSON válido no equivale a elegir el plan correcto.

In [10]:
view.show_outcomes(records)

Prueba · tope,Condición,H0: aciertos,H elegida: aciertos,V final,Llamadas sin error,Cortes,Tokens totales
Test 4 · 2.048,B1,3/8,H1: 7/8,0%,43/45,1,52511
Test 4 · 2.048,B2,3/8,H1: 7/8,0%,62/64,1,78624
Test 4 · 4.096,B1,3/8,H2: 7/8,0%,53/53,0,68762
Test 4 · 4.096,B2,3/8,H1: 7/8,0%,63/65,0,84475
Test 5 · 4.096,B1,3/8,H2: 7/8,0%,53/53,0,68762
Test 5 · 4.096,B2,3/8,H1: 7/8,0%,73/73,0,91579


## 11. Resultados y límites

- **Inferencia:** el perfil con razonamiento cumple en el control con política manual.
- **Automejora inicial (2.048):** B1 y B2 conservan H1; pasan de 3/8 a 7/8 decisiones correctas en test.
- **Adaptación:** B2 reutiliza tres cambios de φ, sin ventaja final en aciertos.
- **Salida:** 4.096 tokens elimina cortes; JSON restringido elimina errores de formato en las ejecuciones registradas. Se mantiene 7/8.
- **Longitud:** más tokens en las candidatas no implica mejor puntuación.

Una búsqueda por condición y pocos escenarios sintéticos. Falta repetir con casos nuevos
para estudiar estabilidad y generalización.

## Fuentes y ejecución

Celdas ejecutadas sobre registros conservados. [Ejecución técnica](../utils/execution.md)
para repetir el control con modelos. Los notebooks individuales conservan el detalle de cada búsqueda.

In [11]:
view.show_sources(records)